# NeuroGraph · Phase 7 · QLoRA fine-tune of the university assistant
Settings: Accelerator **GPU T4 x2**, Internet **On**. Input: the **neurograph-60-intake** notebook output (redacted university chunks). Secrets (**Add-ons → Secrets**, attached): `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`.
Also runs locally (a checkout with its `.venv` and a gitignored `.env`): the setup cells detect Kaggle and otherwise skip the clone, the installs and the Secrets. Locally the notebook 60 output is found under `NEUROGRAPH_INPUT_ROOT` (stands in for `/kaggle/input`).

**Saved outputs:** executed on 2026-10-22 on a local RTX 5060 Ti 16 GB (not on Kaggle), commit `55f60c9`; input = the saved run of notebook 60. The last cell was re-run at `4224df3` after a fix to how the smoke-test answer is logged (the GGUF itself was unchanged). The demo and the Phase 9 evaluation use the 2026-10-21 model. Comparison with the first run: `reports/results.md` §7, repeat run.

## 1. Protect preinstalled packages

In [1]:
from importlib.metadata import version

def write_constraints(pkgs: list[str], path: str) -> None:
    with open(path, "w") as f:
        f.write("\n".join(f"{p}=={version(p)}" for p in pkgs))

write_constraints(["torch", "numpy", "scipy", "scikit-learn", "mne"], "/tmp/keep.txt")
print(open("/tmp/keep.txt").read())

torch==2.11.0+cu128
numpy==2.4.6
scipy==1.18.1
scikit-learn==1.7.2
mne==1.13.2


## 2. Get the latest code + install (adds peft + bitsandbytes for 4-bit training) on Kaggle, or use this checkout (local)

In [2]:
import os, sys
KAGGLE = os.path.isdir("/kaggle/working")
if KAGGLE:
    R, PY = "/kaggle/working/neurograph", "python"
    if not os.path.isdir(R):
        !git clone -q https://github.com/stuBBornButterfly/neurograph.git {R}
    !git -C {R} pull -q
else:  # local run: this checkout and its .venv (packages already installed)
    R, PY = os.environ.get("NEUROGRAPH_REPO") or os.path.abspath(".."), sys.executable
!git -C {R} log --oneline -1
%cd {R}
if KAGGLE:
    !pip install -q -r requirements.txt -c /tmp/keep.txt
    !pip install -q -r requirements-finetune.txt -c /tmp/keep.txt
    !pip install -q -e . -c /tmp/keep.txt

4224df3 (HEAD -> main) export_gguf: capture the llama-cli smoke-test answer inside notebooks (new llama.cpp writes to the terminal when it has one)


/home/accesx-denied/Ayesha/NeuroGraph/NeuroGraph/neurograph


## 3. AuraDB credentials from Kaggle Secrets or `.env` (never printed) + notebook 60 output into the working folder

In [3]:
import glob, shutil
if KAGGLE:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for key in ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"]:
        os.environ[key] = secrets.get_secret(key)
    D, inp = "/kaggle/working/derivatives", "/kaggle/input"
else:  # local run: the gitignored .env (Neo4j login + NEUROGRAPH_* folders; variables already set win)
    from dotenv import load_dotenv
    load_dotenv(f"{R}/.env")
    D, inp = os.environ["NEUROGRAPH_DERIVATIVES_DIR"], os.environ["NEUROGRAPH_INPUT_ROOT"]
src = os.path.dirname(glob.glob(f"{inp}/**/institutional/chunks.jsonl", recursive=True)[0])
if os.path.realpath(src) != os.path.realpath(f"{D}/institutional"):
    shutil.copytree(src, f"{D}/institutional", dirs_exist_ok=True)
print("loaded:", [k for k in ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"] if os.environ.get(k)], "| chunks from", src)

loaded: ['NEO4J_URI', 'NEO4J_USERNAME', 'NEO4J_PASSWORD'] | chunks from /home/accesx-denied/Ayesha/NeuroGraph/NeuroGraph/neurograph/data/notebook_runs/neurograph-60-intake/derivatives/institutional


## 4. Step 7.1: Q&A pairs from the university chunks (Qwen2.5-7B teacher, NLI filter, held-out chunks)

In [4]:
!cd {R} && {PY} -m scripts.make_qa_dataset 2>&1 | grep -v -i -e warn -e it/s -e Loading

== step 7.1 Q&A dataset from university pages | commit 55f60c9 ==
teacher Qwen/Qwen2.5-7B-Instruct (4-bit NF4) | chunks 99 | pairs parsed 548 | NLI-kept 427 (78%, answer entailed by its chunk) | generation 336 s
test pairs dropped because a neighbouring training chunk states the same answer: 16
split  negative
test   False        61
       True         10
train  False       285
       True         55
chunks: train 76 | test 20 (no chunk in both: True)
  Q: How many letter grades can a student earn based on their performance in a course?
  A: Five letter grades: A+, A, A-, B+, B, B-, C+, C, D, and F. [3]
  Q: What are the numerical equivalents for the grade A+?
  A: The numerical equivalent for the grade A+ is 80% and above with 4.00 grade points. [1]
  Q: What is the grade point for a grade of D?
  A: The grade point for a grade of D is 2.00. [1]
time 421 s | saved: /home/accesx-denied/Ayesha/NeuroGraph/NeuroGraph/neurograph/data/notebook_runs/neurograph-70-qlora/derivatives/qlora/trai

## 5. Step 7.2: QLoRA training (4-bit Qwen2.5-3B + LoRA adapters)

In [5]:
!cd {R} && {PY} -m scripts.train_qlora 2>&1 | grep -v -i -e warn -e it/s -e Loading

{'loss': '0.2636', 'grad_norm': '0.9493', 'learning_rate': '0.0001853', 'epoch': '0.4819'}
{'loss': '0.1616', 'grad_norm': '0.733', 'learning_rate': '0.0001588', 'epoch': '0.7229'}
{'loss': '0.1949', 'grad_norm': '0.7215', 'learning_rate': '0.0001233', 'epoch': '0.9639'}



 61%|██████    | 22/36 [00:17<00:14,  1.00s/it]


 50%|█████     | 21/42 [08:49<07:33, 21.62s/it]


100%|██████████| 42/42 [16:58<00:00, 20.96s/it]{'eval_loss': '0.1977', 'eval_runtime': '29.95', 'eval_samples_per_second': '2.371', 'eval_steps_per_second': '1.202', 'epoch': '1'}
{'loss': '0.109', 'grad_norm': '0.4841', 'learning_rate': '8.436e-05', 'epoch': '1.193'}
{'loss': '0.09434', 'grad_norm': '0.2959', 'learning_rate': '4.775e-05', 'epoch': '1.434'}
{'loss': '0.0729', 'grad_norm': '0.4401', 'learning_rate': '1.91e-05', 'epoch': '1.675'}
{'loss': '0.09479', 'grad_norm': '0.3479', 'learning_rate': '2.763e-06', 'epoch': '1.916'}



100%|██████████| 42/42 [17:28<00:00, 24.96s/it]


{'eval_loss': '0.1934', 'eval_runtime': '29.6', 'eval_samples_per_second': '2.399', 'eval_steps_per_second': '1.216', 'epoch': '2'}
{'train_runtime': '1048', 'train_samples_per_second': '0.631', 'train_steps_per_second': '0.04', 'train_loss': '0.2051', 'epoch': '2'}
== step 7.2 QLoRA fine-tune | commit 55f60c9 ==
base Qwen/Qwen2.5-3B-Instruct in 4-bit NF4 | LoRA r 16, alpha 32 on 7 projections | trainable 29.9 M of 1729 M parameters (1.73%)
examples train 331 / test 71 (dropped as too long: 9 / 0) | epochs 2 | effective batch 16 | steps 42
train loss first 0.679 -> last 0.095
  epoch 1: held-out loss 0.198
  epoch 2: held-out loss 0.193
peak GPU memory 12.1 GB
time 1056 s | saved: /home/accesx-denied/Ayesha/NeuroGraph/NeuroGraph/neurograph/data/notebook_runs/neurograph-70-qlora/derivatives/qlora/adapter


## 6. Step 7.3: base vs fine-tuned on held-out chunks + model card

In [6]:
!cd {R} && {PY} -m scripts.eval_qlora 2>&1 | grep -v -i -e warn -e it/s -e Loading

== step 7.3 base vs QLoRA on held-out chunks | commit 55f60c9 ==
test examples 71 (10 with the answer removed) from 20 chunks never seen in training | same prompt, greedy decoding
            token_f1  faithful  cites_right_passage  wrongly_refuses  says_not_in_passages   words  sec_per_answer
base           0.596     0.656                0.295            0.016                   0.7  26.732           0.660
fine-tuned     0.701     0.820                0.902            0.033                   1.0  24.535           0.935

Q: When are Grade Reports mailed to guardians?
  reference: Grade Reports are recorded and prepared by the Office of the Controller of Examinations and mailed to guardians soon after the end of each semester.
  base: [2] Grade Reports are recorded and prepared by the Office of the Controller of Examinations and mailed to guardians soon after the end of each semester.
  fine-tuned: Grade Reports are recorded and prepared by the Office of the Controller of Examinations an

## 7. The whole agent with the fine-tuned model (same 24 questions as notebook 60): adapter on every route, then on the policy route only (the one it was trained on)

In [7]:
!cd {R} && {PY} -m scripts.run_agent_eval --adapter {D}/qlora/adapter --tag _qlora 2>&1 | grep -v -i -e warn -e it/s -e Loading
!cd {R} && {PY} -m scripts.run_agent_eval --adapter {D}/qlora/adapter --adapter-policy-only --tag _qlora_policy 2>&1 | grep -v -i -e warn -e it/s -e Loading

== agent eval (safety -> router -> retrieval/graph -> LLM -> verifier) | commit 55f60c9 ==
LLM Qwen/Qwen2.5-3B-Instruct + QLoRA adapter | graph Neo4j | NLI cross-encoder/nli-deberta-v3-small | load 16 s

kind       n route ok  fact ok verified  sec/q
policy    11     1.00     0.82     0.91    1.7
safety     6     1.00     1.00      nan    0.2
study      4     1.00     0.75     0.50    1.2
literature   3     1.00     1.00     0.67    4.2
all       24     1.00     0.88

safety: 6/6 distress/crisis messages routed to referral; non-safety questions sent to referral: 0
misses: a05 (fact), a08 (fact), a19 (fact)
  a05 answer: 'Below 40% is F. [1]\n\nSources: https://www.university.example/grades-rules-and-regulations'
  a08 answer: 'The passages above do not answer this. Please check the official university website or ask the relevant university office.\n\nSources: https://www.university.example/storage/app/media/Policies/Artificial%20Intelligence%20Policy.pdf'
  a19 answer: 'Our EEG data do

== agent eval (safety -> router -> retrieval/graph -> LLM -> verifier) | commit 55f60c9 ==
LLM Qwen/Qwen2.5-3B-Instruct + QLoRA adapter (policy route only) | graph Neo4j | NLI cross-encoder/nli-deberta-v3-small | load 14 s

kind       n route ok  fact ok verified  sec/q
policy    11     1.00     0.82     0.91    1.6
safety     6     1.00     1.00      nan    0.2
study      4     1.00     1.00     0.50    0.7
literature   3     1.00     1.00     0.67    2.3
all       24     1.00     0.92

safety: 6/6 distress/crisis messages routed to referral; non-safety questions sent to referral: 0
misses: a05 (fact), a08 (fact)
  a05 answer: 'Below 40% is F. [1]\n\nSources: https://www.university.example/grades-rules-and-regulations'
  a08 answer: 'The passages above do not answer this. Please check the official university website or ask the relevant university office.\n\nSources: https://www.university.example/storage/app/media/Policies/Artificial%20Intelligence%20Policy.pdf'

--- a01 [policy, veri

## 8. Step 7.4: merge the adapter into the base + Q4_K_M GGUF for the CPU demo (llama.cpp)
Optional upload to a Hugging Face model repo: add the secrets `HF_TOKEN` (write token) and `HF_REPO` (e.g. `<user>/neurograph-qwen2.5-3b-gguf`).

In [8]:
if KAGGLE:  # optional; without them the GGUF stays in the notebook output (local: HF_TOKEN / HF_REPO from .env, if set)
    for key in ["HF_TOKEN", "HF_REPO"]:
        try:
            os.environ[key] = secrets.get_secret(key)
        except Exception:
            pass
!cd {R} && {PY} -m scripts.export_gguf 2>&1 | grep -v -i -e warn -e it/s -e Loading

== step 7.4 merge + GGUF export | commit 4224df3 ==
merged Qwen/Qwen2.5-3B-Instruct + adapter -> Q4_K_M GGUF 1.93 GB (f16 6.18 GB)
llama.cpp CPU smoke test (4 threads, <= 60 tokens, 4 s incl. load): 'The lowest passing grade is a D. [1]'
not uploaded (HF_TOKEN / HF_REPO not set)
time 92 s | saved: /home/accesx-denied/Ayesha/NeuroGraph/NeuroGraph/neurograph/data/notebook_runs/neurograph-70-qlora/derivatives/qlora/neurograph-qwen2.5-3b-Q4_K_M.gguf
